# PINN — 2D Poisson Equation

$$-\nabla^2 u(x,y) = f(x,y), \quad (x,y) \in [0,1]^2$$

**Forcing term:** $f(x,y) = 2\pi^2 \sin(\pi x)\sin(\pi y)$

**BC (Dirichlet):** $u = 0$ on all four edges of the unit square

**Exact solution:** $u(x,y) = \sin(\pi x)\sin(\pi y)$

This is a steady-state (time-independent) 2-D elliptic PDE.  
Unlike the other notebooks in this repo, there is **no time dimension** — the  
collocation points span $(x,y)$ only, and no initial condition is needed.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import torch
import numpy as np
import matplotlib.pyplot as plt
from pinns import PINN

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Exact Solution

We verify the analytical form before generating data.

In [ ]:
def u_exact(x, y):
    """Exact solution: u(x,y) = sin(pi*x)*sin(pi*y)"""
    return np.sin(np.pi * x) * np.sin(np.pi * y)

def f_source(x, y):
    """Forcing term: f = 2*pi^2 * sin(pi*x)*sin(pi*y)"""
    return 2 * np.pi**2 * np.sin(np.pi * x) * np.sin(np.pi * y)

# Verify: -Laplacian(u_exact) == f_source  (numerically)
# u_xx = -pi^2 sin(pi x) sin(pi y),  u_yy = -pi^2 sin(pi x) sin(pi y)
# -(u_xx + u_yy) = 2 pi^2 sin(pi x) sin(pi y)  ✓
x0, y0 = 0.3, 0.7
lap = -(-np.pi**2 * u_exact(x0, y0) + -np.pi**2 * u_exact(x0, y0))
assert np.isclose(lap, f_source(x0, y0)), 'Analytical check failed!'
print('Exact-solution sanity check passed.')
print(f'u(0.5, 0.5) = {u_exact(0.5, 0.5):.6f}  (expected 1.0)')

## 2. Data Generation

In [ ]:
N_pde = 10000   # interior collocation points
N_bc  = 400     # boundary points (100 per edge)

# --- Interior collocation points (Latin Hypercube-style: just uniform random) ---
x_col = np.random.uniform(0, 1, (N_pde, 1))
y_col = np.random.uniform(0, 1, (N_pde, 1))
xy_pde_np = np.hstack([x_col, y_col])            # (N_pde, 2)
x_pde = torch.tensor(xy_pde_np, dtype=torch.float32, device=device)

# --- Boundary condition: u = 0 on all four edges ---
n_per_edge = N_bc // 4
s = np.linspace(0, 1, n_per_edge)[:, None]       # parameter along each edge

# bottom (y=0), top (y=1), left (x=0), right (x=1)
bc_bottom = np.hstack([s,                     np.zeros_like(s)])
bc_top    = np.hstack([s,                     np.ones_like(s) ])
bc_left   = np.hstack([np.zeros_like(s),      s              ])
bc_right  = np.hstack([np.ones_like(s),       s              ])

xy_bc_np = np.vstack([bc_bottom, bc_top, bc_left, bc_right])  # (N_bc, 2)
x_bc = torch.tensor(xy_bc_np,  dtype=torch.float32, device=device)
u_bc = torch.zeros(N_bc, 1,    dtype=torch.float32, device=device)

print(f'PDE collocation : {x_pde.shape}')
print(f'BC points       : {x_bc.shape}  target: {u_bc.shape}')

## 3. PDE Residual

$$\mathcal{R} = u_{xx} + u_{yy} + f(x,y) = 0$$

In [ ]:
def poisson_residual(model, xy):
    """
    2-D Poisson residual: u_xx + u_yy + f(x,y) = 0
    xy : [N, 2]  (x, y),  requires_grad=True
    """
    u = model(xy)                                       # [N, 1]

    # First derivatives
    grads = torch.autograd.grad(
        u, xy,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]                                                # [N, 2]
    u_x = grads[:, 0:1]
    u_y = grads[:, 1:2]

    # Second derivatives
    u_xx = torch.autograd.grad(
        u_x, xy,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True
    )[0][:, 0:1]

    u_yy = torch.autograd.grad(
        u_y, xy,
        grad_outputs=torch.ones_like(u_y),
        create_graph=True
    )[0][:, 1:2]

    # Forcing term evaluated at collocation points
    x_vals = xy[:, 0:1]
    y_vals = xy[:, 1:2]
    f = 2 * (torch.pi ** 2) * torch.sin(torch.pi * x_vals) * torch.sin(torch.pi * y_vals)

    return u_xx + u_yy + f

## 4. Model & Training

In [ ]:
model = PINN(
    layers=[2, 64, 64, 64, 64, 1],
    activation='tanh',
    residual=False,
    skip=True,
    use_fourier=False,
    use_ntk=False,
).to(device)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
model.fit(
    pde_fn=poisson_residual,
    x_pde=x_pde,
    x_bc=x_bc,
    u_bc=u_bc,
    # No IC — steady-state problem
    epochs=15000,
    lr=1e-3,
    w_pde=1.0,
    w_bc=10.0,    # stronger BC enforcement for Dirichlet
    ntk_adaptive=False,
    log_every=500,
    save_path='./checkpoints/best_poisson.pt',
)

## 5. Loss History

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(model.loss_history['total'], label='Total',  lw=1.5)
ax.semilogy(model.loss_history['pde'],   label='PDE',    lw=1)
ax.semilogy(model.loss_history['bc'],    label='BC',     lw=1)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss — 2D Poisson PINN')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Prediction vs. Exact Solution

In [ ]:
# Build uniform evaluation grid
N_eval = 100
x_lin = np.linspace(0, 1, N_eval)
y_lin = np.linspace(0, 1, N_eval)
XX, YY = np.meshgrid(x_lin, y_lin)           # (N_eval, N_eval)

U_ref = u_exact(XX, YY)                       # (N_eval, N_eval)

model.load_state_dict(torch.load('./checkpoints/best_poisson.pt', weights_only=True))
model.eval()

xy_test = torch.tensor(
    np.stack([XX.ravel(), YY.ravel()], axis=1),
    dtype=torch.float32, device=device
)
with torch.no_grad():
    U_pred = model(xy_test).cpu().numpy().reshape(N_eval, N_eval)

U_err = np.abs(U_pred - U_ref)

# --- Plot ---
vabs = U_ref.max()
kw = dict(origin='lower', extent=[0, 1, 0, 1], aspect='equal', vmin=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(U_ref,  cmap='viridis', vmax=vabs, **kw)
axes[0].set_title('Exact $u(x,y)$')
axes[0].set_xlabel('$x$'); axes[0].set_ylabel('$y$')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(U_pred, cmap='viridis', vmax=vabs, **kw)
axes[1].set_title('PINN $\\hat{u}(x,y)$')
axes[1].set_xlabel('$x$'); axes[1].set_ylabel('$y$')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(U_err,  cmap='hot', vmin=0, **kw)
axes[2].set_title('Absolute Error $|u - \\hat{u}|$')
axes[2].set_xlabel('$x$'); axes[2].set_ylabel('$y$')
plt.colorbar(im2, ax=axes[2])

plt.suptitle('2D Poisson Equation: PINN vs. Exact', fontsize=13)
plt.tight_layout()
plt.show()

rel_l2 = np.linalg.norm(U_pred - U_ref) / (np.linalg.norm(U_ref) + 1e-10)
print(f'Relative L2 error: {rel_l2:.4e}')

## 7. Cross-section Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Slice at y = 0.5  (horizontal midline)
y_idx = N_eval // 2
axes[0].plot(x_lin, U_ref[y_idx],  'k-',  lw=2,   label='Exact')
axes[0].plot(x_lin, U_pred[y_idx], 'b--', lw=1.5, label='PINN')
axes[0].set_title('Slice at $y = 0.5$')
axes[0].set_xlabel('$x$'); axes[0].set_ylabel('$u$')
axes[0].legend()

# Slice at x = 0.5  (vertical midline)
x_idx = N_eval // 2
axes[1].plot(y_lin, U_ref[:, x_idx],  'k-',  lw=2,   label='Exact')
axes[1].plot(y_lin, U_pred[:, x_idx], 'b--', lw=1.5, label='PINN')
axes[1].set_title('Slice at $x = 0.5$')
axes[1].set_xlabel('$y$'); axes[1].set_ylabel('$u$')
axes[1].legend()

plt.suptitle('2D Poisson — Cross-section Comparison', fontsize=12)
plt.tight_layout()
plt.show()